In [5]:
from pygmid import Lookup as lk
import numpy as np
import scipy.constants as sc
import pandas as pd

# read data for GF180MCU NMOS and PMOS device
# the range of channel lengths is 0.28 to 3
# the range for VGS, VDS is 0 to 3.3
n = lk('../../bmurmann_gmoverid/gf180mcuD/simulation/nfet_03v3.mat')
p = lk('../../bmurmann_gmoverid/gf180mcuD/simulation/pfet_03v3.mat')

In [ ]:
#2 Stage Op Amp sizing
GBW = 450e6 #Op Amp bandwidth
CL = 10e-15 #Op Amp load capacitance
gm_id0 = 12
gm_id1 = 12
gm_id2 = 12
l0 = 0.5
l1 = 0.5
l2 = 0.5

gm_1 = GBW*CL*2*np.pi

In [7]:
def get_w(mos_type, gm, gm_id_spec, L_spec, vds, title='specs'):
    
    #Define the MOS type
    if mos_type == 0:
        mos_file = '../../bmurmann_gmoverid/gf180mcuD/simulation/nfet_03v3.mat'
    elif mos_type == 1:
        mos_file = '../../bmurmann_gmoverid/gf180mcuD/simulation/pfet_03v3.mat'
    else:
        print(f"MOS type: 0: NMOS 3.3V, 1: PMOS 3.3")
        return
    print(f"[Transistor: {title}]")
    print(f"gm: {round(gm*1e3,3)} mS")
    # the gm_gds we look up and calculate gds from it
    mos = lk(mos_file)
    vgs = mos.look_upVGS(GM_ID=gm_id_spec, L=L_spec, VDS=vds, VSB=0.0)
    vgs = mos.look_upVGS(GM_ID=gm_id_spec, L=L_spec, VDS=vgs, VSB=0.0)
    print('V_GS =', round(float(vgs), 3), 'V')

    # the gm_gds we look up and calculate gds from it
    gm_gds = mos.lookup('GM_GDS', GM_ID=gm_id_spec, L=L_spec, VDS=vgs, VSB=0)
    gds = gm / gm_gds
    print('gds =', round(gds/1e-9, 2), 'nS')

    # find f_T (which is not stored directly, but we can find the gm to gate capacitance ratio)
    gm_cgg = mos.lookup('GM_CGG', GM_ID=gm_id_spec, L=L_spec, VDS=vgs, VSB=0)
    f_T = gm_cgg / (2*np.pi)
    print('Cgg =', round(gm/gm_cgg/1e-15, 1), 'fF')
    print('f_T =', round(f_T/1e6, 1), 'MHz')

    # find the W of the diode transistor
    id_spec = gm/gm_id_spec
    print('Id =', round(id_spec*1000000, 2), 'uA, rounded Id =', round(id_spec*2000000)/2, 'uA')
    id_w = mos.lookup('ID_W', GM_ID=gm_id_spec, L=L_spec, VDS=vgs, VSB=0)
    w = id_spec / id_w
    print('W =', round(w, 2), 'um, rounded W =', round(w*2)/2, 'um')      

    return [gds,id_spec]

In [9]:
gds_1,id_1 = get_w(0,0.415e-3,10,0.8,1.65,"M1 & M2")

[Transistor: M1 & M2]
gm: 0.415 mS
V_GS = 0.837 V
gds = 1617.34 nS
Cgg = 37.5 fF
f_T = 1760.9 MHz
Id = 41.5 uA, rounded Id = 41.5 uA
W = 15.13 um, rounded W = 15.0 um
